# Hakam — overnight training (v5)

Runs by itself for about 4–5 hours and saves to Drive after every step.

1. **Runtime → Change runtime type → A100 GPU** (High-RAM on)
2. **Runtime → Run all**, approve the Drive popup once
3. Keep this tab open and the laptop awake (plugged in, sleep off)

Results land in `MyDrive/hakam_colab/runs` (`final_v3`, `overnight_report.json`) and `MyDrive/hakam_colab/contracts_v3`.

## 1. GPU

In [ ]:
import torch
name = torch.cuda.get_device_name(0)
print(name)
if "A100" not in name:
    print("WARNING: not an A100 - this will take much longer")

## 2. Drive, code, data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!test -d /content/hakam || git clone -q https://github.com/FerasMad/hakam.git /content/hakam
%cd /content/hakam
!git pull -q && git log --oneline -1
!python scripts/colab_setup.py
!pip install -q transformers

## 3. Cache every camera view (≈7 min, skipped if present)

In [ ]:
!python scripts/cache_frames.py --splits train valid test --all-views --size 224 --tag mv 2>&1 | grep -v clips/s

## 4. Restore round 4 models from Drive
The 5-epoch ensemble that scored card 0.606 / offence 0.610 / body part 0.678 on test. The overnight selection can reuse or beat it.

In [ ]:
import shutil
from pathlib import Path
src, dst = Path("/content/drive/MyDrive/hakam_colab/runs"), Path("/content/hakam/artifacts/runs")
for run in ["mv10_multitask", "mv12_s7", "mv12_s13", "mv13_attr"]:
    if (src / run / "valid_scores.npz").exists() and not (dst / run / "valid_scores.npz").exists():
        shutil.copytree(src / run, dst / run, dirs_exist_ok=True)
        print("restored", run)
    else:
        print("skipped", run)

## 5. Two long runs, in parallel on one A100

| | long1_f9_mix | long2_f4_llrd |
|---|---|---|
| Epochs | 60 (best valid epoch kept) | 60 (best valid epoch kept) |
| Batch | 32 | 32 |
| Trainable blocks | last 3 of 12 | last 8 of 12 |
| LR backbone / heads | 2e-5 / 2e-3 | 4e-5 top block, ×0.75 per block down / 2e-3 |
| Warmup → cosine | 3 epochs | 5 epochs |
| Weight decay | 0.1 | 0.2 |
| Head dropout | 0.3 | 0.5 |
| Mixup α | 0.2 | 0.4 |
| EMA decay | 0.999 | 0.9995 |
| Label smoothing | 0.1 | 0.1 |
| Soft targets | borderline card 0.5, Between offence 0.5 | same |

Optimiser AdamW, mixed precision, every camera view as a sample, views averaged per incident at evaluation. About 1h 45m.

In [ ]:
!mkdir -p logs && (nohup python -u -m src.models.train_mt --tasks card offence action_class body_part --backbone videomae_base --soft-borderline --soft-between --epochs 60 --batch-size 32 --lr 2e-5 --head-lr 2e-3 --weight-decay 0.1 --label-smoothing 0.1 --head-dropout 0.3 --mixup 0.2 --ema 0.999 --warmup-epochs 3 --freeze-blocks 9 --num-workers 6 --select best --predict-splits test --save --name long1_f9_mix > logs/long1.log 2>&1 &) && (nohup python -u -m src.models.train_mt --tasks card offence action_class body_part --backbone videomae_base --soft-borderline --soft-between --epochs 60 --batch-size 32 --lr 4e-5 --head-lr 2e-3 --llrd 0.75 --weight-decay 0.2 --label-smoothing 0.1 --head-dropout 0.5 --mixup 0.4 --ema 0.9995 --warmup-epochs 5 --freeze-blocks 4 --num-workers 6 --select best --predict-splits test --save --name long2_f4_llrd > logs/long2.log 2>&1 &) && echo launched

## 6. Monitor (re-run any time)

In [ ]:
import time; time.sleep(300)
!for f in logs/*.log; do echo "== $f"; grep -E 'epoch [0-9]+:|selected|===|Error|out of memory' $f | tail -n 4 | cut -c1-230; done
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv

## 7. Overnight queue (blocks until done, ≈3 h)

Waits for both runs, then: flip test-time augmentation for every model → a third long run (best recipe, 80 epochs, seed 7) → per-task ensemble selection on **validation only** → test scored once into `final_v3` + contracts → refit of the best single recipe on train+valid (reported separately).

In [ ]:
!python scripts/overnight.py --long long1_f9_mix long2_f4_llrd --extra-name long3_seed7 --extra-args "--epochs 80 --seed 7" --final-name final_v3 --contracts contracts_v3 --refit-name refit_v3 2>&1 | grep -vE 'step [0-9]+/|Warning|LOAD REPORT|UNEXPECTED|MISSING|MISMATCH|^Key|^---|^Notes|^- |num_labels|Loading weights|downloading|reconstructing'

## 8. Results

In [ ]:
!python scripts/summarise_runs.py | tail -60
!cat artifacts/runs/overnight_report.json | head -120